# Diagnostic control — seasonal-naive baseline

**This notebook is not an attempt at a good score.** It is a measurement.

Our LightGBM model scores **0.3966** on a 16-day holdout carved from `train.csv`, and
**0.55331** on the public leaderboard. That 0.157 gap is unexplained. Four hypotheses were
tested and eliminated: window position (late August is not harder — clean penalty −0.023),
store closures (all 54 stores trade into the test window), a misaligned submission (0.983
log-correlation with this very baseline), and horizon degradation (backtests at the true
window position score 0.393–0.422).

So: is our model bad on the real test window, or is the real test window simply hard?

This submits the **simplest defensible forecast** — for each (store, family), the mean of
the same weekday over the last 8 weeks of training data. On our 2017 holdout it scores
**0.52063**.

| If this scores… | Then |
|---|---|
| **~0.52** | The test window behaves like our holdout, and our model is **worse than trivial** on it. The fault is ours — look at calibration and overfitting. |
| **~0.65+** | The test window is genuinely much harder than anything we can validate on. Our 0.553 is respectable and the gap is the period, not the model. |

Either answer redirects the work, which is why it is worth one submission slot.

Runtime: well under a minute. No model is trained.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

COMP = "store-sales-time-series-forecasting"
REQUIRED = {"train.csv", "test.csv"}

def has_data(p: Path) -> bool:
    try:
        return p.is_dir() and REQUIRED.issubset({f.name for f in p.iterdir() if f.is_file()})
    except OSError:
        return False

def find_data() -> Path:
    root = Path("/kaggle/input")
    if root.is_dir():
        for cand in [root / COMP, *sorted(d for d in root.iterdir() if d.is_dir())]:
            if has_data(cand):
                return cand
    for cand in (Path("data"), Path("../data"), Path("../../data")):
        if has_data(cand):
            return cand
    import kagglehub
    got = Path(kagglehub.competition_download(COMP))
    if has_data(got):
        return got
    for sub in got.rglob("*"):
        if has_data(sub):
            return sub
    raise FileNotFoundError("competition data not found")

DATA = find_data()
OUT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("submissions")
OUT.mkdir(parents=True, exist_ok=True)
print("data  :", DATA)
print("output:", OUT)

data  : data
output: submissions


In [2]:
DT = {"store_nbr": "int8", "family": "category", "onpromotion": "int32", "sales": "float32"}
train = pd.read_csv(DATA / "train.csv", parse_dates=["date"], dtype=DT)
test = pd.read_csv(DATA / "test.csv", parse_dates=["date"], dtype=DT)

TRAIN_END = train.date.max()
KEY = ["store_nbr", "family"]

# Same weekday, mean of the last 8 weeks. Nothing else.
recent = train[train.date > TRAIN_END - pd.Timedelta(days=56)].copy()
recent["dow"] = recent.date.dt.dayofweek
profile = recent.groupby(KEY + ["dow"], observed=True).sales.mean()

t = test.assign(dow=test.date.dt.dayofweek)
pred = t.set_index(KEY + ["dow"]).index.map(profile).to_numpy(dtype=float)
pred = np.clip(np.nan_to_num(pred), 0, None)

print(f"fitted on {recent.date.min():%Y-%m-%d} .. {TRAIN_END:%Y-%m-%d}")
print(f"series profiled : {len(profile):,}")
print(f"predictions     : {len(pred):,}   mean {pred.mean():,.2f}   "
      f"median {np.median(pred):,.2f}   zeros {(pred == 0).mean():.1%}")

fitted on 2017-06-21 .. 2017-08-15
series profiled : 12,474
predictions     : 28,512   mean 465.06   median 30.23   zeros 7.4%


In [3]:
submission = pd.DataFrame({"id": test.id, "sales": pred})

assert len(submission) == len(test),   "row count must match test.csv"
assert submission.sales.notna().all(), "every test row needs a prediction"
assert (submission.sales >= 0).all(),  "RMSLE is undefined for negative predictions"
assert submission.id.equals(test.id),  "ids must stay in test.csv order"

submission.to_csv(OUT / "submission.csv", index=False)
print(f"written -> {(OUT / 'submission.csv').resolve()}")
print(submission.head().to_string(index=False))

written -> G:\Meu Drive\_Portifolio\Store Sales - Time Series Forecasting\submissions\submission.csv
     id    sales
3000888    3.875
3000889    0.000
3000890    3.875
3000891 2438.000
3000892    0.000


---
### Record the score here

Our model on the leaderboard: **0.55331**.
This baseline on our 2017-07-31→08-15 holdout: **0.52063**.

Whatever this returns is the number that tells us which of the two explanations is right.